# Point Predictions
This notebook uses some of the metadata that was provided with the reviews to try to predict the variety

This metadata includes:
- country
- points(score)
- price
- province

Each of these models was trained and tested on the metadata:
- SVM
- K Neighbors
- Decision Tree
- Random Forest
- Hist Gradient Boost
- Gradient Boost

## Filter Data

In [2]:
import pandas as pd

df = pd.read_csv('Wine Reviews Dataset/winemag-data-130k-v2.csv')

varieties = [
    'Pinot Noir',
    'Chardonnay',
    'Cabernet Sauvignon',
    'Red Blend',
    'Bordeaux-style Red Blend',
    'Riesling',
    'Sauvignon Blanc',
    'Syrah',
    'Rosé',
    'Merlot'
]

# Keep only the rows with the varieties listed above
df = df[df['variety'].isin(varieties)]

# Replace the wine names with [WINE] so the model doesn't just learn the names of the wines
df['description'] = df['description'].replace(varieties, '[WINE]')

# Discard columns we won't be using
df = df.drop(columns=['Unnamed: 0', 'description', 'designation', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'winery'])

# Drop rows with NaN values
print(len(df))
df.dropna(subset=['points', 'price', 'variety'], inplace=True)
print(len(df))

df[0:10]

71322
67234


,country,points,price,province,variety
3,US,87,13.0,Michigan,Riesling
4,US,87,65.0,Oregon,Pinot Noir
10,US,87,19.0,California,Cabernet Sauvignon
12,US,87,34.0,California,Cabernet Sauvignon
14,US,87,12.0,California,Chardonnay
15,Germany,87,24.0,Mosel,Riesling
20,US,87,23.0,Virginia,Red Blend
21,US,87,20.0,Oregon,Pinot Noir
23,US,87,22.0,California,Merlot
25,US,87,69.0,California,Pinot Noir


In [3]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['variety'])

X = df.drop(columns=['variety'])
# Normalize points and price
X['points'] = X['points'] / 100
X['price'] = X['price'] / X['price'].max()
# Convert categorical features to one-hot encoding
X = pd.get_dummies(X, columns=['country', 'province'])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

X_train[0:10]

,points,price,country_Argentina,country_Armenia,country_Australia,country_Austria,country_Brazil,country_Bulgaria,country_Canada,country_Chile,...,province_Weinviertel,province_Wellington,province_Western Australia,province_Western Cape,province_Württemberg,province_Zenata,province_Österreichischer Perlwein,province_Österreichischer Sekt,province_Štajerska,province_Župa
53804,0.86,0.007576,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
25127,0.86,0.009697,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1486,0.85,0.014545,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78692,0.83,0.006061,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
114040,0.90,0.007576,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
115666,0.91,0.013636,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15486,0.90,0.022727,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
64095,0.91,0.009091,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
312,0.87,0.012121,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127730,0.88,0.005758,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## SVM

In [6]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the SVM model
svm = LinearSVC(max_iter=10000, dual='auto', C=15.0)
svm.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = svm.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{svm.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{svm.score(X_test, y_test)}")

print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")
print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.480004462044732
Testing accuracy:   0.4759425894251506
Precision:          0.49505147237304103
Recall:             0.4759425894251506
F1:                 0.46531568471968454


## K Neighbors

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the KNN model
knn = KNeighborsClassifier(n_neighbors=30)
knn.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = knn.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{knn.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{knn.score(X_test, y_test)}")

print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")
print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.5241415211854166
Testing accuracy:   0.49676507771250095
Precision:          0.5046516228899282
Recall:             0.49676507771250095
F1:                 0.4930491903945286


## Decision Tree

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Decision Tree model
decisiontree = DecisionTreeClassifier(max_depth=1000, random_state=22)
decisiontree.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = decisiontree.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{decisiontree.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{decisiontree.score(X_test, y_test)}")

print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")
print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Cabernet Sauvignon']
Training accuracy:  0.6187926450629334
Testing accuracy:   0.4854614412136536
Precision:          0.4975612086157045
Recall:             0.4854614412136536
F1:                 0.48361804862472607


## Random Forest

In [40]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Random Forest model
randomforest = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=22)
randomforest.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = randomforest.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{randomforest.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{randomforest.score(X_test, y_test)}")

print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")
print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.5820737352891963
Testing accuracy:   0.5063582955306016
Precision:          0.5273297544369002
Recall:             0.5063582955306016
F1:                 0.49992226130466655


## Hist Gradient Boosting
Runs much faster than Gradient Boosting but with worse results

In [36]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Hist Gradient Boosting model
histgradientboosting = HistGradientBoostingClassifier(max_depth=20, max_iter=100, random_state=2)
histgradientboosting.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = histgradientboosting.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{histgradientboosting.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{histgradientboosting.score(X_test, y_test)}")

print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")
print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Chardonnay']
Training accuracy:  0.5282689125625151
Testing accuracy:   0.5064326615601993
Precision:          0.5196074862245289
Recall:             0.5064326615601993
F1:                 0.4996097626847121


## Gradient Boosting

In [41]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Create and train the Gradient Boosting model
gradientboosting = GradientBoostingClassifier(n_estimators=100, max_depth=20, random_state=2)
gradientboosting.fit(X_train, y_train)

# Predict the variety for the test set
y_pred = gradientboosting.predict(X_test)

# Decode the predicted variety
predicted_variety = le.inverse_transform(y_pred)

# Print the predicted variety
print(predicted_variety)

# Print the accuracy scores
print(f"{'Training accuracy: ':<20}{gradientboosting.score(X_train, y_train)}")
print(f"{'Testing accuracy: ':<20}{gradientboosting.score(X_test, y_test)}")

print(f"{'Precision:':<20}{precision_score(y_test, y_pred, average='weighted')}")
print(f"{'Recall:':<20}{recall_score(y_test, y_pred, average='weighted')}")
print(f"{'F1:':<20}{f1_score(y_test, y_pred, average='weighted')}")

['Chardonnay' 'Syrah' 'Riesling' ... 'Sauvignon Blanc' 'Red Blend'
 'Cabernet Sauvignon']
Training accuracy:  0.6184951754141336
Testing accuracy:   0.4917081876998587
Precision:          0.5004412376143635
Recall:             0.4917081876998587
F1:                 0.48972513244934207
